In [19]:
import pandas as pd

url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
df = pd.read_csv(url)

df.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [21]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   crim     506 non-null    float64
 1   zn       506 non-null    float64
 2   indus    506 non-null    float64
 3   chas     506 non-null    int64  
 4   nox      506 non-null    float64
 5   rm       506 non-null    float64
 6   age      506 non-null    float64
 7   dis      506 non-null    float64
 8   rad      506 non-null    int64  
 9   tax      506 non-null    int64  
 10  ptratio  506 non-null    float64
 11  b        506 non-null    float64
 12  lstat    506 non-null    float64
 13  medv     506 non-null    float64
dtypes: float64(11), int64(3)
memory usage: 55.5 KB


crim       0
zn         0
indus      0
chas       0
nox        0
rm         0
age        0
dis        0
rad        0
tax        0
ptratio    0
b          0
lstat      0
medv       0
dtype: int64

In [23]:
X = df.drop("medv", axis=1)
y = df["medv"]

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(random_state=42))
])

In [29]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', RandomForestRegressor(random_state=42))])

In [31]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

preds = pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("RMSE:", rmse)
print("R2 Score:", r2)

RMSE: 2.8129602438238144
R2 Score: 0.8920995891343227


In [35]:
from sklearn.model_selection import cross_val_score
import numpy as np

cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='r2')

print("Cross Validation R2 Scores:", cv_scores)
print("Mean CV R2:", np.mean(cv_scores))

Cross Validation R2 Scores: [0.77212171 0.85784633 0.74149057 0.46989345 0.29456095]
Mean CV R2: 0.6271826004977064


In [38]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='r2')

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV R2:", grid.best_score_)

Best Parameters: {'model__max_depth': None, 'model__n_estimators': 100}
Best CV R2: 0.8262602063852473


In [39]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42)
}

for name, model in models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    
    scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')
    print(name, "Mean R2:", np.mean(scores))

LinearRegression Mean R2: 0.35327592439588223
RandomForest Mean R2: 0.6271826004977064
GradientBoosting Mean R2: 0.6749275463037916


In [42]:
best_model = grid.best_estimator_

importances = best_model.named_steps['model'].feature_importances_

for feature, importance in zip(X.columns, importances):
    print(feature, importance)

crim 0.03806177389300335
zn 0.0017561531081692824
indus 0.00795268115540783
chas 0.0010042649334272775
nox 0.015543774442661404
rm 0.5038449320965203
age 0.013839944097221907
dis 0.06054906975478233
rad 0.0038109087576628006
tax 0.01566064484820767
ptratio 0.01631341163943569
b 0.012153615631364977
lstat 0.3095088256421353
